# MP33 — Satellite Preprocessing Pipeline

[Brief](README.md) · [Validation](VALIDATION.md). Objective: select bands, preserve validity, fit normalization only on training data, and extract correctly georeferenced patches. Prerequisite MP32. This lesson uses a cached aerial training crop and does not change the frozen SR pipeline.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
from affine import Affine
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'stage_08_remote_sensing/MP33_satellite_preprocessing_pipeline'))
from mp33_tools import select_valid_bands,fit_standardization,standardize,extract_patches


## 1. Bands and validity before arithmetic

CHW means channels/bands, rows, columns. Band indices are zero-based; select [0,1,2,3] for the manifest's red/green/blue/NIR order, or [3,0,1] for an explicitly labeled false-colour display. Reordering is not color-space conversion. Use a CHW boolean mask so nodata can differ by band; intersect with finite values. Never infer nodata from darkness alone. The selection helper computes in float64 to avoid unsigned subtraction overflow.

Invalid normalized values are stored as zero **with a false mask**. Zero is a neutral fill, not an observation. Downstream losses and statistics must use the mask; partial patches are rejected by default.

In [ ]:
x=np.array([[[1.,3.],[99.,5.]],[[2.,2.],[99.,2.]]])
mask=np.ones_like(x,dtype=bool);mask[:,1,0]=False
selected,valid=select_valid_bands(x,mask,[1,0])
assert selected[0,0,0]==2 and not valid[0,1,0]


## 2. Train-only standardization

For valid training samples in band b, μ_b=Σx/N_b and σ_b=√(Σ(x−μ_b)²/N_b), using population variance. Transform with z=(x−μ_b)/s_b, where s_b=σ_b unless σ_b=0, in which case s_b=1. Save these statistics and reuse them without fitting on validation or test. An all-invalid band cannot be fitted. The explicit split guard prevents accidental validation fitting, but cannot verify that a caller's label is truthful: preserve provenance too.

Hand example [1,3,5]: μ=3, σ=√(8/3)≈1.63299, z≈[−1.224745,0,1.224745]. A constant band [2,2,2] maps to zero. Values outside [0,1] are normal for z-scores. Unlike fixed DN/10000, z-scores are centered per band and alter spectral-vector angles; they are not physically meaningful reflectance or a replacement for the frozen model's expected inputs.

In [ ]:
stats=fit_standardization(x,mask,split='train')
z,valid=standardize(x,mask,stats)
assert np.allclose(z[0][valid[0]],[-np.sqrt(1.5),0,np.sqrt(1.5)])
assert np.all(z[1]==0) and z.dtype==np.float32
shifted,_=standardize(x+10,mask,stats)
print('Frozen train statistics:',stats)
print('Shifted-data mean (not refitted):',shifted[0][valid[0]].mean())
try:fit_standardization(x,mask,split='validation')
except ValueError as error:print('Expected guard:',error)
else:raise AssertionError('Validation fitting must fail')


## 3. Patch coordinates and acceptance

For a crop starting at column c₀,row r₀, the new affine transform is T_crop=T_parent×Translation(c₀,r₀). Keep the CRS and band semantics unchanged. In a north-up 10-m fixture whose corner is (100,200), column2,row0 starts at (120,200). Pixel centers still add half a pixel. Rotated transforms use the full affine multiplication; do not guess north from the picture.

The extractor walks full nonoverlapping P×P patches in row-major order. Acceptance requires the fraction of pixels valid in **all selected bands** to exceed the threshold (default1). It reports rejected patches and incomplete bottom/right margins. Patching must follow geographic scene splitting, never precede it.

In [ ]:
grid=np.arange(2*5*5).reshape(2,5,5);valid=np.ones_like(grid,dtype=bool);valid[:,0,0]=False
patches,discarded=extract_patches(grid,valid,Affine(10,0,100,0,-10,200),size=2)
assert len(patches)==3 and discarded['rejected_full_patches']==1
assert tuple(patches[0]['transform'])[:6]==(10,0,120,0,-10,200)
print('First accepted patch:',patches[0]['row'],patches[0]['column'],patches[0]['transform'])
print('Explicit discarded margins:',discarded)


## 4. Measured cached-raster experiment

The experiment first checks the synthetic fixture, verifies the SHA-256 of the NAIP training raster, and reads a 128×128 four-band window at (238,238). Statistics are fitted on this training window only, then saved. Normalized output is float32; patch extraction uses float64 working arrays. Neither the original file nor model preprocessing is modified. Re-run with `python experiment.py` only when needed; this notebook reads the recorded result.

In [ ]:
record=json.loads((root/'reports/mp33-preprocessing.json').read_text())
assert record['source']['split']=='train' and record['statistics']['fit_split']=='train'
assert record['accepted_patches']==16
assert np.allclose(record['normalized_band_means'],0,atol=1e-5)
assert np.allclose(record['normalized_band_stds'],1,atol=1e-5)
for band,stats,mean,std in zip(record['band_order'],record['statistics']['bands'],record['normalized_band_means'],record['normalized_band_stds']):
    print(band,'stored-DN mean/std',stats['mean'],stats['std'],'normalized mean/std',mean,std)
print('First patch metadata:',record['patches'][0])
print('License:',record['license'])


## 5. Exercises and solutions

1. Why must a false mask accompany a zero fill? 2. What are μ and σ for [1,3,5], and why is validation recentering leakage? 3. For the 10-m transform, compute the upper-left corner of a patch at row2,column2. 4. Is per-band z-scoring appropriate before physical spectral-angle comparison?

<details><summary>Solutions</summary>

1. Otherwise fabricated zeros affect losses/statistics. 2. μ=3 and σ=√(8/3); validation fitting changes the learned preprocessing using held-out information and masks distribution shift. 3. (120,180), with unchanged CRS and 10-m spacing. 4. No: different band offsets/scales change spectral directions. Use a declared scientifically compatible physical protocol instead.
</details>

## Interpretation and connection

Sixteen full 32×32 patches were accepted from the cached crop, with no invalid pixels there. Near-zero means/unit population standard deviations confirm the declared transform; they do **not** demonstrate improved SR quality. Synthetic masking tests cover rejection paths absent from this crop. The experiment preserves CRS and derives every patch transform. Next MP34 builds matched HR/LR pairs with explicit degradation. Limits: one aerial crop, no cloud-mask validation, no reflectance calibration, no test fitting, no learner-mastery claim.